# 00 - Latent Space Notebook Configuration

Shared Python configuration for the cancer latent-space notebooks.

Run this once near the top of Notebooks 1-6:

```python
%run ./00_latent_space_config.ipynb
```

This notebook centralizes paths, metadata contracts, loader helpers, and label/abbreviation dictionaries so downstream notebooks do not each maintain their own versions.


In [7]:
from pathlib import Path
import pandas as pd
import numpy as np


## Project root and canonical directories


In [8]:
NOTEBOOK_DIR = Path.cwd()


def find_project_root(start_dir: Path) -> Path:
    """Locate the global-cancer-complexity project root from a notebook directory."""
    candidates = [start_dir] + list(start_dir.parents)
    for candidate in candidates:
        if (candidate / "data" / "global_cancer").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate global-cancer-complexity project root from notebook directory: "
        f"{start_dir}"
    )


PROJECT_DIR = find_project_root(NOTEBOOK_DIR)

# R preprocessing export location
ML_INPUT_DIR = PROJECT_DIR / "data" / "global_cancer" / "processed" / "ml_inputs"

# Python latent-space handoff location
PROCESSED_DIR = PROJECT_DIR / "data" / "global_cancer" / "processed"

# Canonical project output locations
OUTPUT_DIR = PROJECT_DIR / "output" / "global_cancer"
PLOTS_DIR = OUTPUT_DIR / "plots"
TABLES_DIR = OUTPUT_DIR / "tables"
TRAINING_PLOTS_DIR = PLOTS_DIR / "training"
LATENT_PLOTS_DIR = PLOTS_DIR / "latent"
LATENT_TABLES_DIR = TABLES_DIR / "latent"
MODELS_DIR = OUTPUT_DIR / "models" / "latent"

# Notebook-specific output directories. These keep downstream products organized
# without changing the canonical parent directories used by the reporting pipeline.
NB04_PLOTS_DIR = LATENT_PLOTS_DIR / "notebook_04"
NB04_TABLES_DIR = LATENT_TABLES_DIR / "notebook_04"
NB05_PLOTS_DIR = LATENT_PLOTS_DIR / "notebook_05"
NB05_TABLES_DIR = LATENT_TABLES_DIR / "notebook_05"
NB06_PLOTS_DIR = LATENT_PLOTS_DIR / "notebook_06"
NB06_TABLES_DIR = LATENT_TABLES_DIR / "notebook_06"

for path in [
    PROCESSED_DIR,
    TRAINING_PLOTS_DIR,
    LATENT_PLOTS_DIR,
    LATENT_TABLES_DIR,
    NB04_PLOTS_DIR,
    NB04_TABLES_DIR,
    NB05_PLOTS_DIR,
    NB05_TABLES_DIR,
    NB06_PLOTS_DIR,
    NB06_TABLES_DIR,
    MODELS_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

print("Notebook directory :", NOTEBOOK_DIR)
print("Project directory  :", PROJECT_DIR)
print("ML input directory :", ML_INPUT_DIR)
print("Processed directory:", PROCESSED_DIR)
print("Output directory   :", OUTPUT_DIR)


Notebook directory : C:\Users\drziy\OneDrive\Desktop\Ali\scientific_research_current\global-cancer-complexity\projects\cancer-latent-space\notebooks
Project directory  : C:\Users\drziy\OneDrive\Desktop\Ali\scientific_research_current\global-cancer-complexity
ML input directory : C:\Users\drziy\OneDrive\Desktop\Ali\scientific_research_current\global-cancer-complexity\data\global_cancer\processed\ml_inputs
Processed directory: C:\Users\drziy\OneDrive\Desktop\Ali\scientific_research_current\global-cancer-complexity\data\global_cancer\processed
Output directory   : C:\Users\drziy\OneDrive\Desktop\Ali\scientific_research_current\global-cancer-complexity\output\global_cancer


## Dataset and file names


In [9]:
CHIP_ID = "hu35ksuba"
FILTER_METHOD = "variance"
TOP_N = 3000

EXPRESSION_FILENAME = f"{CHIP_ID}_expr_top{TOP_N}_{FILTER_METHOD}.csv"
METADATA_FILENAME = f"{CHIP_ID}_metadata_aligned.csv"
FEATURES_FILENAME = f"{CHIP_ID}_top{TOP_N}_{FILTER_METHOD}_features.csv"

EXPRESSION_PATH = ML_INPUT_DIR / EXPRESSION_FILENAME
METADATA_PATH = ML_INPUT_DIR / METADATA_FILENAME
FEATURES_PATH = ML_INPUT_DIR / FEATURES_FILENAME

# Notebook 2 outputs
X_TRAIN_PATH = PROCESSED_DIR / "X_train.npy"
X_VAL_PATH = PROCESSED_DIR / "X_val.npy"
X_TEST_PATH = PROCESSED_DIR / "X_test.npy"

IDS_TRAIN_PATH = PROCESSED_DIR / "ids_train.npy"
IDS_VAL_PATH = PROCESSED_DIR / "ids_val.npy"
IDS_TEST_PATH = PROCESSED_DIR / "ids_test.npy"

X_SCALED_ALL_PATH = PROCESSED_DIR / "X_scaled_all.npy"
SAMPLE_IDS_ALL_PATH = PROCESSED_DIR / "sample_ids_all.npy"
FEATURE_IDS_PATH = PROCESSED_DIR / "feature_ids.npy"

METADATA_ALIGNED_PATH = PROCESSED_DIR / "metadata_aligned.csv"
EXPRESSION_ALIGNED_PATH = PROCESSED_DIR / "expression_aligned.csv"
SCALER_STATS_PATH = PROCESSED_DIR / "scaler_stats.csv"

# Notebook 3 outputs
LATENT_PATH = PROCESSED_DIR / "latent.npy"
MODEL_STATE_DICT_PATH = MODELS_DIR / "first_vae_state_dict.pt"
MODEL_CHECKPOINT_PATH = MODELS_DIR / "first_vae_full_checkpoint.pt"

print("Expression path:", EXPRESSION_PATH, "| exists:", EXPRESSION_PATH.exists())
print("Metadata path  :", METADATA_PATH, "| exists:", METADATA_PATH.exists())
print("Features path  :", FEATURES_PATH, "| exists:", FEATURES_PATH.exists())
print("Latent path    :", LATENT_PATH, "| exists:", LATENT_PATH.exists())


Expression path: C:\Users\drziy\OneDrive\Desktop\Ali\scientific_research_current\global-cancer-complexity\data\global_cancer\processed\ml_inputs\hu35ksuba_expr_top3000_variance.csv | exists: True
Metadata path  : C:\Users\drziy\OneDrive\Desktop\Ali\scientific_research_current\global-cancer-complexity\data\global_cancer\processed\ml_inputs\hu35ksuba_metadata_aligned.csv | exists: True
Features path  : C:\Users\drziy\OneDrive\Desktop\Ali\scientific_research_current\global-cancer-complexity\data\global_cancer\processed\ml_inputs\hu35ksuba_top3000_variance_features.csv | exists: True
Latent path    : C:\Users\drziy\OneDrive\Desktop\Ali\scientific_research_current\global-cancer-complexity\data\global_cancer\processed\latent.npy | exists: True


## Metadata contract


In [10]:
REQUIRED_METADATA_COLUMNS = [
    "sample_id",
    "geo_accession",
    "title",
    "platform_id",
    "disease_clean",
    "tissue_clean",
    "condition",
    "tissue_label",
]

MINIMAL_METADATA_COLUMNS = [
    "sample_id",
    "geo_accession",
    "condition",
    "tissue_label",
]


## Shared comparison labels and plotting abbreviations

`COMPARISON_MAP` maps cleaned tissue/disease pairs to the project-level comparison labels used in the R pipeline and Quarto reports. `PLOT_LABEL_MAP` maps those project labels to compact visual labels.


In [11]:
COMPARISON_MAP = {
    ("Bladder", "bladder transitional cell carcinoma"): "BLAD/TCC",
    ("Breast", "breast adenocarcinoma"): "BR/BRAD",
    ("Colon", "colorectal adenocarcinoma"): "COL/COADREAD",
    ("Kidney", "renal cell carcinoma"): "KID/RCC",
    ("Lung", "lung adenocarcinoma"): "LU/LUAD",
    ("Ovary", "ovarian adenocarcinoma"): "OV/OVAD",
    ("Pancreas", "pancreatic adenocarcinoma"): "PA/PAAD",
    ("Prostate", "prostate adenocarcinoma"): "PR/PRAD",
    ("Uterus", "uterine adenocarcinoma"): "UT/EAC",
    ("Brain", "glioblastoma"): "Brain/GBM",
    ("Brain", "medulloblastoma"): "Brain/MB",
    ("Lymphoid Tissue", "Follicular lymphoma"): "GC/FL",
    ("Lymphoid Tissue", "large B-cell lymphoma"): "GC/LBCL",
    ("Blood", "acute myeloid leukemia"): "PB/AML",
    ("Bone Marrow", "B-cell ALL"): "PB/B-ALL",
    ("Bone Marrow", "T-cell ALL"): "PB/T-ALL",
}

PLOT_LABEL_MAP = {
    "BLAD/TCC": "TCC",
    "BR/BRAD": "BRAD",
    "COL/COADREAD": "COADREAD",
    "KID/RCC": "RCC",
    "LU/LUAD": "LUAD",
    "OV/OVAD": "OVAD",
    "PA/PAAD": "PAAD",
    "PR/PRAD": "PRAD",
    "UT/EAC": "EAC",
    "Brain/GBM": "GBM",
    "Brain/MB": "MB",
    "GC/FL": "FL",
    "GC/LBCL": "LBCL",
    "PB/AML": "AML",
    "PB/B-ALL": "B-ALL",
    "PB/T-ALL": "T-ALL",
}

STATE_CANCER = "cancer"
STATE_NORMAL = "normal"


## Loader, alignment, and labeling helpers


In [12]:
def load_expression(expression_path: Path = EXPRESSION_PATH) -> pd.DataFrame:
    """Load expression matrix as samples x features, indexed by sample_id."""
    if not expression_path.exists():
        raise FileNotFoundError(f"Expression file not found: {expression_path}")
    expression = pd.read_csv(expression_path, index_col=0)
    expression.index.name = "sample_id"
    return expression


def load_metadata(
    metadata_path: Path = METADATA_PATH,
    require_full_contract: bool = True,
) -> pd.DataFrame:
    """Load metadata while preserving sample_id as both index and column."""
    if not metadata_path.exists():
        raise FileNotFoundError(f"Metadata file not found: {metadata_path}")

    metadata = pd.read_csv(metadata_path)

    if "sample_id" not in metadata.columns:
        raise AssertionError(
            "Metadata is missing 'sample_id'. "
            "Check export_ml_inputs.R and avoid pd.read_csv(..., index_col=0)."
        )

    metadata = metadata.set_index("sample_id", drop=False)

    required = REQUIRED_METADATA_COLUMNS if require_full_contract else MINIMAL_METADATA_COLUMNS
    missing = [c for c in required if c not in metadata.columns]
    if missing:
        raise AssertionError(f"Missing required metadata columns: {missing}")

    return metadata


def validate_expression_metadata_alignment(
    expression: pd.DataFrame,
    metadata: pd.DataFrame,
) -> None:
    """Ensure expression rows and metadata rows are exactly aligned by sample_id."""
    if expression.shape[0] != metadata.shape[0]:
        raise AssertionError(
            f"Sample count mismatch: expression has {expression.shape[0]}, "
            f"metadata has {metadata.shape[0]}"
        )

    if not expression.index.equals(metadata.index):
        first_bad = next(
            (i for i, (a, b) in enumerate(zip(expression.index, metadata.index)) if a != b),
            None,
        )
        raise AssertionError(
            "Expression and metadata indices are not identically ordered. "
            f"First mismatch at position {first_bad}: "
            f"{expression.index[first_bad]} != {metadata.index[first_bad]}"
        )


def load_ml_inputs(require_full_contract: bool = True):
    """Load expression and metadata, then validate sample alignment."""
    expression = load_expression()
    metadata = load_metadata(require_full_contract=require_full_contract)
    validate_expression_metadata_alignment(expression, metadata)
    return expression, metadata


def load_feature_table(features_path: Path = FEATURES_PATH) -> pd.DataFrame:
    """Load top-variance feature table generated by preprocessing."""
    if not features_path.exists():
        raise FileNotFoundError(f"Feature table not found: {features_path}")
    return pd.read_csv(features_path)


def load_latent_array(latent_path: Path = LATENT_PATH) -> np.ndarray:
    """Load latent coordinate matrix generated by Notebook 3."""
    if not latent_path.exists():
        raise FileNotFoundError(f"Latent file not found: {latent_path}")
    return np.load(latent_path)


def latent_columns(df: pd.DataFrame) -> list[str]:
    """Return latent coordinate columns in z1, z2, ... order."""
    return sorted(
        [c for c in df.columns if c.startswith("z")],
        key=lambda c: int(c[1:]) if c[1:].isdigit() else c,
    )


def build_latent_dataframe(
    latent_path: Path = LATENT_PATH,
    metadata_path: Path = METADATA_ALIGNED_PATH,
    require_full_contract: bool = True,
) -> pd.DataFrame:
    """Load latent coordinates and metadata, assert row alignment, and return joined table."""
    latent = load_latent_array(latent_path)
    metadata = load_metadata(metadata_path=metadata_path, require_full_contract=require_full_contract)

    if latent.shape[0] != metadata.shape[0]:
        raise AssertionError(
            f"Latent/metadata sample mismatch: {latent.shape[0]} latent rows vs "
            f"{metadata.shape[0]} metadata rows"
        )

    latent_df = pd.DataFrame(latent, index=metadata.index)
    latent_df.columns = [f"z{i + 1}" for i in range(latent_df.shape[1])]
    latent_df.index.name = "sample_id"

    if not latent_df.index.equals(metadata.index):
        raise AssertionError("Latent rows and metadata rows are not aligned.")

    if not metadata["sample_id"].astype(str).equals(metadata.index.to_series().astype(str)):
        raise AssertionError("sample_id column does not match metadata index.")

    return latent_df.join(metadata)


def add_standard_latent_labels(
    df: pd.DataFrame,
    comparison_map: dict = COMPARISON_MAP,
    plot_label_map: dict = PLOT_LABEL_MAP,
) -> pd.DataFrame:
    """Add tissue/disease/state/comparison/short_label/plot_label columns."""
    out = df.copy()
    out["tissue"] = out["tissue_clean"]
    out["disease"] = out["disease_clean"]
    out["state"] = out["condition"]

    cancer_mask = out["state"].eq(STATE_CANCER)
    out["comparison"] = pd.NA
    out.loc[cancer_mask, "comparison"] = (
        out.loc[cancer_mask, "tissue"].astype(str)
        + "/"
        + out.loc[cancer_mask, "disease"].astype(str)
    )

    out["short_label"] = pd.NA
    out.loc[cancer_mask, "short_label"] = [
        comparison_map.get((t, d))
        for t, d in zip(out.loc[cancer_mask, "tissue"], out.loc[cancer_mask, "disease"])
    ]

    out["plot_label"] = pd.NA
    out.loc[cancer_mask, "plot_label"] = out.loc[cancer_mask, "short_label"].map(plot_label_map)

    return out


def report_unmapped_latent_labels(df: pd.DataFrame) -> None:
    """Display unmapped cancer labels for project or plot-level abbreviations."""
    cancer = df[df["state"].eq(STATE_CANCER)].copy()

    missing_project = cancer[cancer["short_label"].isna()]
    if missing_project.empty:
        print("All cancer labels mapped to project comparison labels.")
    else:
        print("Unmapped cancer labels detected for project comparison labels:")
        display(missing_project[["tissue", "disease"]].drop_duplicates())

    missing_plot = cancer[cancer["plot_label"].isna()]
    if missing_plot.empty:
        print("All project labels mapped to plot labels.")
    else:
        print("Unmapped cancer labels detected for plot labels:")
        display(missing_plot[["short_label", "tissue", "disease"]].drop_duplicates())


def build_latent_working_table(df: pd.DataFrame) -> pd.DataFrame:
    """Return standardized table consumed by Notebooks 5 and 6."""
    base_cols = [
        "geo_accession",
        "title",
        "platform_id",
        "tissue",
        "disease",
        "state",
        "comparison",
        "short_label",
        "plot_label",
        "tissue_label",
    ]
    available_base_cols = [c for c in base_cols if c in df.columns]
    return df[available_base_cols + latent_columns(df)].copy()


## Usage pattern

```python
%run ./00_latent_space_config.ipynb

df = build_latent_dataframe()
df = add_standard_latent_labels(df)
latent_working = build_latent_working_table(df)
```
